# LCE Review Pipeline — 01 · Ingest + Clean (Bronze → Silver)

Pulls Google reviews from the Uberall API for all store locations, lands them as a raw
**bronze** Delta table, then dedups / null-handles / text-cleans into a **silver** table.

**Medallion layers written by this notebook**

| Layer  | Table            | Contents                                             |
|--------|------------------|------------------------------------------------------|
| Bronze | `reviews_bronze` | Raw API pull, one row per review, no transforms      |
| Silver | `reviews_silver` | Deduped, nulls handled, URLs/whitespace stripped     |

Downstream, `02_enrich` reads `reviews_silver` and writes `reviews_gold`.

> **Env:** set the `catalog` / `schema` widgets. Defaults target the **test** schema
> `jdub_demo.little_caesars`; a Job task overrides them to prod `ioc_sandbox.ai_strategy`.

In [ ]:
# --- Parameters (widgets) ---------------------------------------------------
# Test default: jdub_demo.little_caesars   |   Prod (Job override): ioc_sandbox.ai_strategy
dbutils.widgets.text("catalog", "jdub_demo", "Catalog")
dbutils.widgets.text("schema", "little_caesars", "Schema")
# Store the Uberall API key as a Databricks secret; the widget lets a Job pass a scope/key.
dbutils.widgets.text("uberall_secret_scope", "", "Uberall secret scope (optional)")
dbutils.widgets.text("uberall_secret_key", "", "Uberall secret key (optional)")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()
FQ = lambda t: f"{CATALOG}.{SCHEMA}.{t}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Target: {CATALOG}.{SCHEMA}")
print(f"  bronze -> {FQ('reviews_bronze')}")
print(f"  silver -> {FQ('reviews_silver')}")

## Bronze — Uberall ingestion

Pulls paginated Google reviews across all store locations into one DataFrame, saved as-is
to `reviews_bronze` before any cleaning.

> **Secret handling:** the API key is read from a Databricks secret when the
> `uberall_secret_*` widgets are set (recommended for Jobs). It falls back to an inline
> constant only for interactive dev — do not commit a real key.

In [ ]:
import requests, pandas as pd, time

# Prefer a secret; fall back to inline only for interactive dev.
_scope = dbutils.widgets.get("uberall_secret_scope").strip()
_key   = dbutils.widgets.get("uberall_secret_key").strip()
if _scope and _key:
    API_KEY = dbutils.secrets.get(scope=_scope, key=_key)
else:
    # DEV FALLBACK ONLY — replace with a secret before scheduling. Do not commit a real key.
    API_KEY = "REPLACE_WITH_UBERALL_API_KEY"

URL = "https://uberall.com/api/data-points"
HEADERS = {"X-API-KEY": API_KEY}

LOCATIONS = [
    "4856515","5225743","5136010","4928801","5125657","4602855","4649654","5248505","4586212","4565543",
    "4893654","4647342","5072097","5072086","5248504","4661741","5072096","5125656","4812782","4954475",
    "5260290","4540957","4540960","5133674","2704035","5082278","4540938","4576722","4540928","4579851",
    "4552612","4573349","4592804","4920226","4595296","5072089","4540912","4607668","4605940","4731721",
    "4926505","4583921","4626803","5125658","4549005","5133672","4623998","4540910","5125663","4742361",
    "4540974","4540944","4602856","4729559","5821160","5161715","5893215","4540971","5072093","4540919",
    "4729560","4540923","4726848","4551916","5735005","4699810","4753668","5118723","4656242","4540955",
    "4937443","4540956","4584787","4713230","4729562","4946214","4551915","5133671","4595297","4901705",
    "4586211","4540933","5166082","5325174","4540947","5072091","5072094","5895842","4599674","4540913",
    "4540935","5506843","4590051","5072087","5191018","4540930","5150021","4586210","4540941","5072085",
]

PAGE_SIZE = 50
all_reviews = []
for loc_id in LOCATIONS:
    page, loc_total = 0, 0
    while True:
        query = {"locationIds": loc_id, "text": "true", "max": str(PAGE_SIZE), "page": str(page),
                 "dataPointTypes": "REVIEW", "directoryTypes": "GOOGLE"}
        resp = requests.get(URL, headers=HEADERS, params=query)
        reviews = resp.json().get("response", {}).get("inbox", {}).get("dataPoints", [])
        for r in reviews:
            all_reviews.append({
                "date": r.get("actionDate"), "rating": r.get("rating"), "review_text": r.get("data"),
                "directoryType": r.get("directoryType"), "type": r.get("type"),
                "directLink": r.get("directLink"), "locationId": r.get("locationId"),
            })
        loc_total += len(reviews)
        time.sleep(0.6)
        if len(reviews) < PAGE_SIZE:
            break
        page += 1
    print(f"  {loc_id}: {loc_total} text reviews pulled")

df = pd.DataFrame(all_reviews)
print(f"\nTotal reviews pulled: {len(df)}")
display(df)

In [ ]:
# Land raw pull as bronze (overwrite full snapshot each run).
spark_df = spark.createDataFrame(df)
(spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(FQ("reviews_bronze")))
print(f"Saved {df.shape[0]} rows to {FQ('reviews_bronze')}")

## Silver — clean

Dedup (exact, then by `review_text` + `date` + `locationId`), handle nulls (flag missing
ratings, fill string nulls), strip URLs / normalize whitespace, and drop rows that are
empty after cleaning. Reads from the bronze DataFrame in memory to avoid stale rows.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import re

# Read from the in-memory pull (avoids stale bronze rows with null directLink, etc.)
df_s = spark.createDataFrame(df)
print(f"Bronze row count: {df_s.count()}")

# Dedup: exact, then on the natural key.
df_s = df_s.dropDuplicates()
df_s = df_s.dropDuplicates(["review_text", "date", "locationId"])
print(f"After dedup: {df_s.count()}")

# Drop empty review_text.
df_s = df_s.filter(F.col("review_text").isNotNull() & (F.trim(F.col("review_text")) != ""))

# Missing-rating flag + fill; string nulls -> 'unknown'.
df_s = (df_s
    .withColumn("rating_missing", F.when(F.col("rating").isNull(), True).otherwise(False))
    .withColumn("rating", F.coalesce(F.col("rating"), F.lit(0.0)))
    .withColumns({c: F.coalesce(F.col(c), F.lit("unknown")) for c in ["directoryType", "type"]}))

# Text clean: strip URLs, collapse whitespace.
@F.udf(StringType())
def clean_text(t):
    if t is None: return None
    t = re.sub(r"http\S+|www\.\S+", "", t)
    return re.sub(r"\s+", " ", t).strip()

df_s = df_s.withColumn("review_text_cleaned", clean_text(F.col("review_text")))
df_s = df_s.filter(F.col("review_text_cleaned").isNotNull() & (F.trim(F.col("review_text_cleaned")) != ""))

df_silver = df_s.select(
    F.col("date"), F.col("rating"), F.col("rating_missing"),
    F.col("review_text_cleaned").alias("review_text"),
    F.col("directoryType"), F.col("type"), F.col("locationId"))

print(f"Silver row count: {df_silver.count()}")
df_silver.printSchema()
display(df_silver.limit(15))

In [ ]:
(df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(FQ("reviews_silver")))
print(f"Saved {df_silver.count()} rows to {FQ('reviews_silver')}")